# SNC - Thesis Results (measured across versions)

Evaluates **every model version you list** and turns the measured
numbers into publication-quality figures and a multi-sheet Excel
workbook, all saved into a tidy folder tree on Drive.

Run it top-to-bottom and leave it - results are written to Drive
after each version, so a disconnect never loses finished work.

## Pipeline
1. Mount Drive, clone `feature/without-fsd50k`, link data + models
2. Bring the decoded-dataset cache to SSD (fast mixer loads)
3. Set `VERSIONS = [...]` and measure each one (SI-SDR + detection)
4. Build comparison + per-class figures and the Excel from the
   measured results

## What lands on Drive
`{DRIVE_ROOT}/thesis_results/`
- `raw/measured_results.json` - every measured number
- `figures/comparison/{png,pdf}/` - F1 and SI-SDRi across versions
- `figures/per_version/<v>/{png,pdf}/` - per-class F1, SI-SDRi,
  precision/recall for each version
- `tables/snc_results.xlsx` - comparison + per-version sheets

> Detection is scored on the locally-validatable classes for every
> version, so the comparison is fair even for older FSD50K-trained
> checkpoints (their phantom-only classes are never surfaced).

## 1. Mount Drive, clone the repo, link data + models

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/snc'
GITHUB_USER = 'keremtutumlu'
GITHUB_REPO_NAME = 'selective-noise-cancelling'
BRANCH = 'feature/without-fsd50k'

# Private repo? Add a Colab Secret named GITHUB_TOKEN (key icon, scope: repo).
import os, subprocess, shutil
from pathlib import Path

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

clone_url = (f'https://{token}@github.com/{GITHUB_USER}/{GITHUB_REPO_NAME}.git'
             if token else
             f'https://github.com/{GITHUB_USER}/{GITHUB_REPO_NAME}.git')

REPO_DIR = Path(f'/content/{GITHUB_REPO_NAME}')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
result = subprocess.run(['git', 'clone', '-b', BRANCH, clone_url, str(REPO_DIR)],
                        capture_output=True, text=True)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('git clone failed - if the repo is private add a '
                       'GITHUB_TOKEN secret in the left sidebar.')
os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
subprocess.run(['git', 'log', '--oneline', '-3'], check=True)

In [ ]:
# Symlink data + saved_models to Drive, and point the cache / log env vars
# at Drive so the decoded-dataset cache and the audit log persist.
drive_data = Path(DRIVE_ROOT) / 'data'
drive_models = Path(DRIVE_ROOT) / 'saved_models'
for local, target in [(REPO_DIR / 'data', drive_data),
                      (REPO_DIR / 'saved_models', drive_models)]:
    if local.is_symlink() or local.exists():
        local.unlink() if local.is_symlink() else shutil.rmtree(local)
    local.symlink_to(target)
    print(f'{local} -> {target}')

os.environ['SNC_DATA_CACHE_DIR'] = '/content'
os.environ['SNC_DRIVE_CACHE_DIR'] = f'{DRIVE_ROOT}/cache'
os.environ['SNC_DRIVE_LOG_DIR'] = f'{DRIVE_ROOT}/run_logs'
print('cache  ->', os.environ['SNC_DRIVE_CACHE_DIR'])
print('logs   ->', os.environ['SNC_DRIVE_LOG_DIR'])

In [ ]:
!pip install -q librosa==0.11.0 soundfile scikit-learn pandas openpyxl
import tensorflow as tf
print('TF version :', tf.__version__)
print('GPUs       :', tf.config.list_physical_devices('GPU'))

## 2. Decoded-dataset cache
Brings the 56-class ESC-50 + UrbanSound8K pickle to the local SSD so
every mixer the evaluators build loads in seconds.

In [ ]:
# Bring the decoded-dataset cache (ESC-50 + UrbanSound8K, 56 classes) to the
# local SSD so every mixer build below loads in seconds instead of re-decoding.
# After the v2.8 training run this pickle is already mirrored on Drive.
!python scripts/prep_data_cache.py --info
!python scripts/prep_data_cache.py

## 3. Configure versions and measurement settings
Edit `VERSIONS` to the checkpoints you have on Drive. Missing ones
are skipped automatically.

In [ ]:
# === What to measure ===
# List the versions whose checkpoints live on Drive at
#   saved_models/separation_models/separator_unet_film_multi_<v>.h5  (+ _classes.json)
# Versions whose files are missing are skipped with a printed note, so it is
# safe to list every version you have ever trained.
VERSIONS = ["v2.1", "v2.2", "v2.3", "v2.4", "v2.6", "v2.8"]

SI_SDR_N = 800      # positive-only mixtures for the SI-SDR evaluation
DET_N    = 200      # multi-class mixtures for the detection evaluation
REMEASURE = False   # True re-measures versions already saved in raw/measured_results.json

import sys
sys.path.insert(0, 'src/model_training')
sys.path.insert(0, 'src/data_preparation')
import model_config as cfg

# Output tree on Drive - survives the runtime, drops straight into the thesis.
#   thesis_results/
#     raw/measured_results.json          every measured number (provenance)
#     figures/comparison/{png,pdf}/      F1 + SI-SDRi across versions
#     figures/per_version/<v>/{png,pdf}/ per-class F1, SI-SDRi, precision/recall
#     tables/snc_results.xlsx            comparison + per-version sheets
OUT = Path(DRIVE_ROOT) / 'thesis_results'
for sub in ['raw', 'figures/comparison/png', 'figures/comparison/pdf', 'tables']:
    (OUT / sub).mkdir(parents=True, exist_ok=True)
print('Output tree:', OUT)
print('Versions to measure:', VERSIONS)

## 4. Measure every version
Runs the real SI-SDR and detection evaluations in-process and saves
the structured results to Drive after each version. Expect roughly
1-3 min per version on an A100.

In [ ]:
# Measure every version in-process. Results are saved to Drive after EACH
# version, so a disconnect or a single bad checkpoint never loses the rest -
# re-running the cell resumes from where it stopped.
import json, gc, time, traceback
import tensorflow as tf
from separation_mixer import SeparationMixer
from evaluate_conditioned_separator import ConditionedSeparatorEvaluator
import evaluate_detection

# Detection is scored on the locally-validatable classes only, identically for
# every version, so the F1 comparison is apples-to-apples (no phantom FSD50K
# false positives from older 227-class checkpoints).
LOCAL_CLASSES = SeparationMixer(cfg.DATA_ROOT, negative_prob=0.0, seed=0).class_names
print(f'Local validatable classes: {len(LOCAL_CLASSES)}')

RAW_JSON = OUT / 'raw' / 'measured_results.json'
measured = json.loads(RAW_JSON.read_text()) if RAW_JSON.exists() else {}
if measured:
    print(f'Resuming - already measured: {list(measured)}')

for v in VERSIONS:
    if v in measured and not REMEASURE:
        print(f'[{v}] already measured - skipping (set REMEASURE=True to redo).')
        continue
    ckpt, classes_file = cfg.model_path(v), cfg.classes_path(v)
    if not ckpt.exists() or not classes_file.exists():
        print(f'[{v}] checkpoint or classes file missing - SKIPPED ({ckpt.name}).')
        continue
    print(f"\n{'=' * 64}\n[{v}] measuring  ({ckpt.name})\n{'=' * 64}")
    try:
        t0 = time.time()
        sisdr = ConditionedSeparatorEvaluator(
            cfg.DATA_ROOT, ckpt, n_test=SI_SDR_N).evaluate()
        det = evaluate_detection.evaluate(
            ckpt, cfg.DATA_ROOT, n_test=DET_N, allowed=LOCAL_CLASSES, version=v)
        measured[v] = {'version': v, 'sisdr': sisdr, 'detection': det,
                       'seconds': round(time.time() - t0, 1)}
        RAW_JSON.write_text(json.dumps(measured, indent=2))  # incremental save
        print(f"[{v}] OK - SI-SDRi {sisdr['si_sdri']:+.2f} dB, "
              f"F1 {det['f1_mean']:.3f}  ({measured[v]['seconds']}s)")
    except Exception:
        print(f'[{v}] FAILED:\n{traceback.format_exc()}')
    finally:
        tf.keras.backend.clear_session()
        gc.collect()

print(f'\nDone. {len(measured)} version(s) measured -> {RAW_JSON}')

## 5. Figures and Excel from the measured results
Everything below reads `measured` (and `raw/measured_results.json`),
so this half can be re-run on its own without re-measuring.

In [ ]:
# Plotting setup + a tidy summary table from the measured results.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

assert measured, ('No measurements found - run the measurement cell first, or '
                  'check that the checkpoints exist on Drive.')

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('ggplot')
plt.rcParams['figure.dpi'] = 110
ACCENT, HILITE, MUTED = '#2563eb', '#dc2626', '#9ca3af'

CMP_PNG = OUT / 'figures' / 'comparison' / 'png'
CMP_PDF = OUT / 'figures' / 'comparison' / 'pdf'


def save_cmp(fig, name):
    fig.savefig(CMP_PNG / f'{name}.png', dpi=300, bbox_inches='tight')
    fig.savefig(CMP_PDF / f'{name}.pdf', bbox_inches='tight')
    print(f'saved: comparison/{name}')


order = {v: i for i, v in enumerate(VERSIONS)}
rows = []
for v, e in measured.items():
    rows.append({
        'version': v,
        'si_sdri_db': e['sisdr']['si_sdri'],
        'detection_f1': e['detection']['f1_mean'],
        'det_tp': e['detection']['tp_total'],
        'det_fp': e['detection']['fp_total'],
        'det_fn': e['detection']['fn_total'],
        'has_det_head': e['detection']['has_detection_head'],
        'n_classes_tested': len(e['detection']['per_class']),
        'seconds': e.get('seconds'),
    })
summary_df = (pd.DataFrame(rows)
              .sort_values('version', key=lambda s: s.map(order))
              .reset_index(drop=True))
summary_df

### Comparison figures - F1 and SI-SDRi across versions

In [ ]:
# Comparison figures: detection F1 and SI-SDRi across the measured versions.
# F1
fig, ax = plt.subplots(figsize=(9, 5))
best = summary_df['detection_f1'].idxmax()
colors = [HILITE if i == best else ACCENT for i in range(len(summary_df))]
bars = ax.bar(summary_df['version'], summary_df['detection_f1'], color=colors)
for b, val in zip(bars, summary_df['detection_f1']):
    ax.text(b.get_x() + b.get_width() / 2, val + 0.005, f'{val:.2f}',
            ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Macro F1')
ax.set_xlabel('Model version')
ax.set_title('Detection F1 across model versions (measured)\n'
             f'identical {list(measured.values())[0]["detection"]["n_test"]}-mixture '
             'test, allow-list = local classes')
ax.set_ylim(0, max(summary_df['detection_f1']) * 1.25 + 1e-3)
save_cmp(fig, 'cmp_detection_f1')
plt.show()

# SI-SDRi
fig, ax = plt.subplots(figsize=(9, 5))
best = summary_df['si_sdri_db'].idxmax()
colors = [HILITE if i == best else ACCENT for i in range(len(summary_df))]
bars = ax.bar(summary_df['version'], summary_df['si_sdri_db'], color=colors)
for b, val in zip(bars, summary_df['si_sdri_db']):
    ax.text(b.get_x() + b.get_width() / 2, val - 0.4, f'{val:.1f}',
            ha='center', va='top', fontsize=9)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('SI-SDRi (dB)')
ax.set_xlabel('Model version')
ax.set_title('Separation SI-SDRi across model versions (measured)')
save_cmp(fig, 'cmp_sisdri')
plt.show()

### Per-version per-class figures
For each measured version: detection F1 per class, SI-SDRi per
class, and a precision-vs-recall scatter.

In [ ]:
# Per-version per-class figures: detection F1, SI-SDRi, precision/recall.
def plot_per_version(v, det_rows, sis_rows):
    pv_png = OUT / 'figures' / 'per_version' / v / 'png'
    pv_pdf = OUT / 'figures' / 'per_version' / v / 'pdf'
    pv_png.mkdir(parents=True, exist_ok=True)
    pv_pdf.mkdir(parents=True, exist_ok=True)

    def _save(fig, name):
        fig.savefig(pv_png / f'{name}.png', dpi=300, bbox_inches='tight')
        fig.savefig(pv_pdf / f'{name}.pdf', bbox_inches='tight')

    # Detection F1 per class.
    det = pd.DataFrame(det_rows).sort_values('f1')
    fig, ax = plt.subplots(figsize=(8, max(3.0, 0.28 * len(det) + 1.5)))
    ax.barh(det['class'], det['f1'], color=plt.cm.RdYlGn(det['f1']))
    ax.axvline(det['f1'].mean(), color='black', linestyle='--', linewidth=1,
               label=f"mean F1 = {det['f1'].mean():.2f}")
    ax.set_xlabel('F1')
    ax.set_title(f'{v} - detection F1 by class')
    ax.legend(loc='lower right')
    _save(fig, f'{v}_detection_f1_per_class')
    plt.show()

    # SI-SDRi per class.
    sis = pd.DataFrame(sis_rows).sort_values('si_sdri')
    fig, ax = plt.subplots(figsize=(8, max(3.0, 0.28 * len(sis) + 1.5)))
    bar_colors = ['#16a34a' if x > -15 else ACCENT if x > -30 else HILITE
                  for x in sis['si_sdri']]
    ax.barh(sis['class'], sis['si_sdri'], color=bar_colors)
    ax.axvline(sis['si_sdri'].mean(), color='black', linestyle='--',
               linewidth=1, label=f"mean = {sis['si_sdri'].mean():.1f} dB")
    ax.set_xlabel('SI-SDRi (dB)')
    ax.set_title(f'{v} - SI-SDRi by class')
    ax.legend(loc='lower left')
    _save(fig, f'{v}_sisdri_per_class')
    plt.show()

    # Precision vs recall.
    det = pd.DataFrame(det_rows)
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    sc = ax.scatter(det['recall'], det['precision'], s=det['n'] * 14,
                    c=det['f1'], cmap='viridis', alpha=0.75,
                    edgecolors='white', linewidths=0.5)
    rr = np.linspace(0.01, 1, 200)
    for f1v in (0.2, 0.4, 0.6, 0.8):
        pp = (f1v * rr) / (2 * rr - f1v)
        m = (pp > 0) & (pp <= 1)
        ax.plot(rr[m], pp[m], color=MUTED, linewidth=0.6, linestyle=':')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f'{v} - precision vs recall per class')
    fig.colorbar(sc, ax=ax, label='F1')
    _save(fig, f'{v}_precision_recall')
    plt.show()


for v, e in measured.items():
    print(f'--- {v} ---')
    plot_per_version(v, e['detection']['per_class'], e['sisdr']['per_class'])

### Excel workbook

In [ ]:
# One Excel workbook: a comparison sheet plus detection + SI-SDR sheets per
# version, written next to the figures on Drive.
xlsx = OUT / 'tables' / 'snc_results.xlsx'
with pd.ExcelWriter(xlsx, engine='openpyxl') as xl:
    summary_df.to_excel(xl, sheet_name='Version comparison', index=False)
    for v, e in measured.items():
        pd.DataFrame(e['detection']['per_class']).to_excel(
            xl, sheet_name=f'{v} detection'[:31], index=False)
        pd.DataFrame(e['sisdr']['per_class']).to_excel(
            xl, sheet_name=f'{v} SI-SDR'[:31], index=False)
print('wrote', xlsx)

## 6. Done - verify saved artifacts

In [ ]:
# Confirm every artifact landed on Drive.
print('Artifacts under', OUT, '\n')
for p in sorted(OUT.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(OUT)}   ({p.stat().st_size // 1024} KB)')